# 네이버 뉴스 '러시아' 기사 — 언론사별 긍정·중립·부정 분류
기간: **2026-10-28**

위에서부터 순서대로 실행하세요. API 키는 필요 없습니다.
결과는 `data/2026-10-28/` 폴더에 날짜별로 따로 저장되어, 다른 날짜 데이터와 섞이지 않습니다.

| 파트 | 내용 | 대상 언론사 |
|---|---|---|
| **Part 1. 공통 코드** | 설정, 수집, 공통 분류 도구 | 두 파트 언론사를 합쳐서 수집 |
| **Part 2. 정치·사회·경제·세계** | 시사 단어 사전으로 분류 (+ 기타 섹션) | `NEWS_PRESS` — 종합지·방송·통신사 |
| **Part 3. 연예·스포츠·생활·문화** | 문화 단어 사전으로 분류 | `CULTURE_PRESS` — 종합지 + 스포츠·연예 전문지 |

Part 1을 한 번 실행한 뒤에는 Part 2, Part 3을 따로따로 다시 돌려도 됩니다.


In [1]:
# %pip install selenium requests beautifulsoup4 pandas tqdm matplotlib

---
# Part 1. 공통 코드
## 1-1. 설정
기간, 검색어, **파트별 섹션·언론사 목록**을 여기서 정합니다. 언론사를 넣고 빼려면 `NEWS_PRESS` / `CULTURE_PRESS`를 고치세요.


In [ ]:
import csv, random, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, timedelta
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from tqdm import tqdm

QUERY = "러시아"
START = END = date(2026, 10, 28)

# ── Part 2 (정치·사회·경제·세계): 종합지·방송·통신사 ──
NEWS_SECTIONS = ["정치", "세계", "경제", "사회", "오피니언"]
NEWS_PRESS = {
    "023": "조선일보", "025": "중앙일보", "020": "동아일보",
    "028": "한겨레", "032": "경향신문", "469": "한국일보",
    "005": "국민일보", "081": "서울신문", "022": "세계일보",
    "001": "연합뉴스", "056": "KBS", "214": "MBC", "055": "SBS",
}

# ── Part 3 (연예·스포츠·생활·문화): 종합지 + 스포츠·연예 전문지 ──
CULTURE_SECTIONS = ["생활", "문화", "연예", "스포츠", "여행"]
CULTURE_PRESS = {
    **NEWS_PRESS,
    # 스포츠지
    "076": "스포츠조선", "382": "스포츠동아", "468": "스포츠서울", "144": "스포츠경향",
    "241": "일간스포츠", "477": "스포티비뉴스",
    # 연예 전문
    "109": "OSEN", "311": "엑스포츠뉴스", "117": "마이데일리", "108": "스타뉴스",
    "609": "뉴스엔", "312": "텐아시아",
}

# 수집은 두 파트 언론사를 합쳐서 한 번에 (섹션은 기사를 열어봐야 알 수 있어서)
TARGET_PRESS = {**NEWS_PRESS, **CULTURE_PRESS}

# 기간별 폴더 → 예전 날짜로 모은 데이터와 섞이지 않음 (노트북이 workplace/ 안에 있어서 상위의 data/ 사용)
DATA = Path("../data") / (START.isoformat() if START == END else f"{START}_{END}")
DATA.mkdir(parents=True, exist_ok=True)
LINKS_CSV    = DATA / "links.csv"
DONE_DAYS    = DATA / "done_days.txt"
ARTICLES_CSV = DATA / "articles.csv"
OUT          = DATA / "report"

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36")
# 일반 뉴스(n.news) + 연예(m.entertain) + 스포츠(m.sports, 종목별 경로 포함)
ARTICLE_RE = re.compile(r"(n\.news|m\.entertain|m\.sports)\.naver\.com/(?:[a-z]+/)?article/(\d{3})/(\d{10})")

print("저장 위치:", DATA.resolve())


## 1-2. 수집 함수 정의


In [2]:
# ───────── 1단계: 검색결과에서 링크 수집 (Selenium) ─────────
def make_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument(f"user-agent={UA}")
    opts.add_argument("--window-size=1280,2000")
    return webdriver.Chrome(options=opts)  # Selenium 4.6+는 크롬드라이버 자동 설치


def search_url(d):
    dot, raw = d.strftime("%Y.%m.%d"), d.strftime("%Y%m%d")
    return ("https://search.naver.com/search.naver?where=news&sm=tab_opt&sort=1&pd=3"
            f"&query={quote(QUERY)}&ds={dot}&de={dot}&nso=so:dd,p:from{raw}to{raw}")


def scroll_to_end(driver, pause=1.2, max_rounds=150):
    """무한스크롤 대응: 페이지 높이가 3번 연속 그대로면 끝"""
    last, stable = 0, 0
    for _ in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause + random.random() * 0.5)
        h = driver.execute_script("return document.body.scrollHeight")
        stable = stable + 1 if h == last else 0
        if stable >= 3:
            break
        last = h


def article_url(m):
    """일반 뉴스는 예전과 같은 형식으로, 연예·스포츠는 원래 주소 그대로"""
    host, oid, aid = m.groups()
    if host == "n.news":
        return f"https://n.news.naver.com/mnews/article/{oid}/{aid}"
    return "https://" + m.group(0)


def grab_links(driver, found):
    for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='n.news.naver.com'], "
                                  "a[href*='m.entertain.naver.com'], a[href*='m.sports.naver.com']"):
        m = ARTICLE_RE.search(a.get_attribute("href") or "")
        if not m:
            continue
        host, oid, aid = m.groups()
        if TARGET_PRESS is None or oid in TARGET_PRESS:
            found[(oid, aid)] = article_url(m)


def links_for_day(driver, d):
    driver.get(search_url(d))
    time.sleep(1.5)
    found = {}
    for _ in range(400):  # 페이지 번호 UI가 나오는 경우도 대응
        scroll_to_end(driver)
        grab_links(driver, found)
        nxt = driver.find_elements(By.CSS_SELECTOR, "a.btn_next[aria-disabled='false']")
        if not nxt:
            break
        nxt[0].click()
        time.sleep(1.5)
    return found


def collect_links(start=START, end=END, headless=True):
    """start~end 날짜를 하루씩 검색해서 data/links.csv에 이어 붙임. 이미 끝난 날짜는 건너뜀"""
    done = set(DONE_DAYS.read_text().split()) if DONE_DAYS.exists() else set()
    days = [start + timedelta(i) for i in range((end - start).days + 1)]
    todo = [d for d in days if d.isoformat() not in done]
    print(f"검색할 날짜 {len(todo)}일 (이미 완료 {len(days) - len(todo)}일)")
    if not todo:
        return

    driver = make_driver(headless)
    new_file = not LINKS_CSV.exists()
    try:
        with open(LINKS_CSV, "a", newline="", encoding="utf-8") as f, \
             open(DONE_DAYS, "a", encoding="utf-8") as dlog:
            w = csv.writer(f)
            if new_file:
                w.writerow(["search_date", "oid", "aid", "url"])
            for d in tqdm(todo, desc="days"):
                try:
                    found = links_for_day(driver, d)
                except Exception as e:
                    print(f"[{d}] 실패: {e}")
                    driver.quit()
                    driver = make_driver(headless)
                    continue
                for (oid, aid), url in found.items():
                    w.writerow([d.isoformat(), oid, aid, url])
                f.flush()
                dlog.write(d.isoformat() + "\n")
                dlog.flush()
                if len(found) < 5:
                    print(f"[{d}] {len(found)}건뿐 — 차단 또는 페이지 구조 변경 여부 확인")
                time.sleep(random.uniform(2, 4))
    finally:
        driver.quit()


# ───────── 2단계: 기사 페이지 파싱 (requests) ─────────
ENT_SPORTS = {"m.entertain.naver.com": "연예", "m.sports.naver.com": "스포츠"}


def parse_date_kr(s):
    """'2026.10.20. 오후 6:56' → '2026-10-20 18:56:00' (일반 뉴스와 같은 형식)"""
    m = re.search(r"(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d{1,2}):(\d{2})", s)
    if not m:
        return ""
    y, mo, d, ampm, h, mi = m.groups()
    h = int(h) % 12 + (12 if ampm == "오후" else 0)
    return f"{y}-{mo}-{d} {h:02d}:{mi}:00"


def parse_article(url, session):
    r = session.get(url, timeout=10)
    r.raise_for_status()
    s = BeautifulSoup(r.text, "html.parser")
    oid = ARTICLE_RE.search(url).group(2)
    host = r.url.split("/")[2]

    def text(sel):
        el = s.select_one(sel)
        return el.get_text(" ", strip=True) if el else ""

    if host in ENT_SPORTS:  # 연예·스포츠는 페이지 구조가 달라서 따로 파싱
        title = text("h2[class*='article_title']")
        press = TARGET_PRESS.get(oid, "") if TARGET_PRESS else ""
        published = parse_date_kr(text("em.date"))
        section = ENT_SPORTS[host]
        body_el = s.select_one("._article_content")
    elif host == "n.news.naver.com":
        title = text("#title_area") or text("h2.media_end_head_headline")

        logo = s.select_one("a.media_end_head_top_logo img")
        press = logo.get("alt", "") if logo else ""
        if not press:
            meta = s.select_one("meta[property='og:article:author']")
            press = meta.get("content", "").split("|")[0].strip() if meta else ""
        if not press and TARGET_PRESS:
            press = TARGET_PRESS.get(oid, "")

        ts = s.select_one("span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME")
        published = ts.get("data-date-time", "") if ts else ""

        section = ",".join(e.get_text(strip=True) for e in s.select("em.media_end_categorize_item"))
        body_el = s.select_one("#dic_area") or s.select_one("#newsct_article")
    else:  # 그 밖의 도메인으로 넘어가면 건너뜀
        return None

    body = ""
    if body_el:
        for junk in body_el.select("em.img_desc, span.end_photo_org, script, style, table"):
            junk.decompose()
        body = re.sub(r"\s+", " ", body_el.get_text(" ", strip=True))

    return {"press": press, "oid": oid, "published": published,
            "section": section, "title": title, "body": body, "url": url}


def parse_articles(workers=4, limit=None):
    """links.csv의 기사를 파싱해 data/articles.csv에 이어 붙임. limit으로 일부만 테스트 가능"""
    links = pd.read_csv(LINKS_CSV, dtype=str).drop_duplicates("url")
    done = set(pd.read_csv(ARTICLES_CSV, dtype=str)["url"]) if ARTICLES_CSV.exists() else set()
    todo = [u for u in links["url"] if u not in done]
    if limit:
        todo = todo[:limit]
    print(f"링크 {len(links)}건 / 이번에 파싱 {len(todo)}건")

    session = requests.Session()
    session.headers["User-Agent"] = UA

    def job(u):
        time.sleep(random.uniform(0.3, 0.8))
        return parse_article(u, session)

    fields = ["press", "oid", "published", "section", "title", "body", "url"]
    new_file = not ARTICLES_CSV.exists()
    with open(ARTICLES_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        if new_file:
            w.writeheader()
        with ThreadPoolExecutor(workers) as ex:
            futs = {ex.submit(job, u): u for u in todo}
            for fut in tqdm(as_completed(futs), total=len(futs), desc="articles"):
                try:
                    row = fut.result()
                except Exception as e:
                    print(f"{futs[fut]} 실패: {e}")
                    continue
                if row:
                    w.writerow(row)
                    f.flush()

NameError: name 'START' is not defined

## 1-3. 수집 실행 + 날짜 확인
`RECOLLECT = True`면 이미 검색한 날짜도 네이버 검색 결과를 **처음부터 다시** 훑어서, 새로 넣은 언론사나 연예·스포츠 기사를 추가로 모읍니다. 이미 파싱한 기사는 건너뛰니 중복으로 쌓이지 않습니다. 다 모은 뒤에는 `False`로 바꿔 두면 다시 실행할 때 검색을 건너뜁니다.


In [3]:
RECOLLECT = True  # True: 이미 검색한 날짜도 검색 결과를 처음부터 다시 훑음 (이미 파싱한 기사는 건너뜀). 다 모은 뒤엔 False로
if RECOLLECT and DONE_DAYS.exists():
    DONE_DAYS.unlink()

collect_links()
parse_articles()

articles = pd.read_csv(ARTICLES_CSV)
dates = pd.to_datetime(articles["published"], errors="coerce").dt.date
print(f"\n총 {len(articles)}건")
print("발행일 분포:")
print(dates.value_counts().sort_index())
if not (dates == START).any():
    print(f"\n⚠️ {START} 기사가 하나도 없습니다. 네이버 검색 결과가 다른 날짜로 나온 것이니 알려주세요.")

검색할 날짜 0일 (이미 완료 1일)
링크 114건 / 이번에 파싱 0건


articles: 0it [00:00, ?it/s]


총 114건
발행일 분포:
published
2026-10-28    114
Name: count, dtype: int64


## 1-4. 공통 분류 도구
두 파트가 함께 쓰는 규칙과 함수입니다.
- 긍정어 바로 뒤에 "해 달라·요구" 등이 붙으면 **중립**(아직 안 된 일), "난항·무산·불투명·(입장이) 달라" 등이 붙으면 **부정**으로 봅니다. 결과에 `종전(요구)`, `종전(난항)`처럼 표시됩니다.
- 북한이 기사 중심이면 남은 긍정어도 부정으로 뒤집습니다 (북한과 러시아의 협력은 한국 입장에서 위협).
- `run_part(이름, 분류함수, 섹션, 언론사)`가 해당 섹션·언론사 기사만 골라 분류하고 표·그래프·CSV를 만듭니다.


In [ ]:
import matplotlib.pyplot as plt
import platform
plt.rcParams["font.family"] = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}.get(platform.system(), "NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

RUSSIA_WORDS = ["러시아", "푸틴", "크렘린", "모스크바", "북러", "한러", "러軍", "러·", "露"]
NK_WORDS = ["북러", "북·러", "북한", "北", "김정은", "평양", "북한군", "조러"]  # '북'만 쓰면 북유럽도 걸려서 제외


# 긍정어 바로 뒤에 붙으면 의미가 바뀌는 표현
REQUEST_RE = re.compile(r"[해어아와워춰줘려]\s?달라")  # "종전해 달라"(요구) ↔ "종전 수단 달라"(다르다) 구분
UNREALIZED = ["요구", "촉구", "해야", "원해", "원한다", "필요", "바란다", "주문", "호소"]   # 아직 안 됨 → 중립
FAILED = ["불투명", "난항", "무산", "멀어", "어려", "어렵", "결렬", "교착", "회의적",
          "실패", "좌초", "제동", "안갯속", "없", "못", "거부", "물 건너",
          "달라", "다르", "이견", "엇갈", "취소", "불발"]                    # 안 되고 있음 → 부정


def pos_hits(title, body, words):
    """긍정어마다 뒤 12글자(문장부호 전까지)를 보고 긍정/중립/부정으로 나눔"""
    pos, neg, neu = {}, {}, {}
    for src, weight in ((title, 2), (body, 1)):
        for w in words:
            for m in re.finditer(re.escape(w), src):
                after = re.split(r"[…,.!?\"'“”‘’\]\)]", src[m.end(): m.end() + 12])[0]
                is_request = REQUEST_RE.search(after)
                if any(x in REQUEST_RE.sub("", after) for x in FAILED):
                    k, d = f"{w}(난항)", neg
                elif is_request or any(x in after for x in UNREALIZED):
                    k, d = f"{w}(요구)", neu
                else:
                    k, d = w, pos
                d[k] = d.get(k, 0) + weight
    return pos, neg, neu


def load_articles():
    a = pd.read_csv(ARTICLES_CSV, dtype=str)
    a["published"] = pd.to_datetime(a["published"], errors="coerce")
    a = a[(a["published"] >= pd.Timestamp(START)) & (a["published"] < pd.Timestamp(END + timedelta(1)))]
    a = a.drop_duplicates("url").drop_duplicates(["press", "title"])
    a["section"] = a["section"].fillna("").str.split(",").str[0].replace("", "미분류")
    a[["title", "body"]] = a[["title", "body"]].fillna("")
    return a.reset_index(drop=True)


def show_part(df, name, k):
    order = ["긍정", "중립", "부정"]
    counts = pd.crosstab(df["press"], df["label"]).reindex(columns=order, fill_value=0)
    counts["합계"] = counts.sum(axis=1)
    pct = (counts[order].div(counts["합계"], axis=0) * 100).round(1)
    counts = counts.sort_values("합계", ascending=False)
    counts.loc["전체"] = counts.sum()

    print("\n[언론사별 기사 수]")
    display(counts)
    print("\n[언론사별 비율 %]")
    display(pct.sort_values("부정", ascending=False))

    pct.sort_values("부정").plot(kind="barh", stacked=True, figsize=(9, max(2, len(pct) * 0.45)),
                                color=["#4c78a8", "#bbbbbb", "#e45756"])
    plt.xlabel("%")
    plt.title(f"[{name}] 한국 언론의 러시아 관련 보도 평가 ({START} ~ {END})")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(OUT / f"press_label_share_{name}.png", dpi=150)
    plt.show()
    counts.to_csv(OUT / f"press_label_counts_{name}.csv", encoding="utf-8-sig")

    # 분류가 맞는지 눈으로 확인용
    for lab in ["부정", "긍정", "중립"]:
        if not (df["label"] == lab).any():
            continue
        print(f"\n[{name} · {lab} 예시]")
        display(df[df["label"] == lab][["press", "section", "score", "title",
                                         "pos_words", "neg_words", "neu_words"]].head(k))


def run_part(name, classify_fn, sections, press, k=3, hide_empty=False):
    """sections(네이버 섹션)와 press(언론사 코드)에 해당하는 기사만 classify_fn으로 분류.
    hide_empty=True면 러시아 관련 기사가 없을 때 아무것도 출력하지 않음"""
    OUT.mkdir(parents=True, exist_ok=True)
    df = load_articles()
    df = df[df["section"].isin(sections) & df["oid"].isin(list(press))]
    if not df.empty:
        df = pd.concat([df, df.apply(lambda r: classify_fn(r["title"], r["body"]), axis=1)], axis=1)
        rel = df[df["relevant"]]
    else:
        rel = df
    if rel.empty and hide_empty:
        return rel

    print(f"━━━━━━━━━━ [{name}] 대상 {len(df)}건 중 러시아 관련 {len(rel)}건 ━━━━━━━━━━")
    if rel.empty:
        return rel
    print("섹션별:", rel["section"].value_counts().to_dict())
    show_part(rel, name, k)
    rel.to_csv(OUT / f"classified_{name}.csv", index=False, encoding="utf-8-sig")
    print("\n결과 폴더:", OUT.resolve())
    return rel


---
# Part 2. 정치·사회·경제·세계 분석
## 2-1. 시사 단어 사전
기준은 **러시아 입장이 아니라 한국 언론이 사안을 어떻게 평가하는가**입니다.
- **긍정**: 협력, 평화, 관계 개선, 경제 기회처럼 한국 언론이 좋게 평가
- **부정**: 위협, 도발, 우려, 리스크처럼 나쁘게 평가. **북한과 러시아의 협력은 부정**
- **중립**: 만났다, 밝혔다처럼 평가 없이 사실만 전달

드론 격추·교전·사상자처럼 **전쟁 피해·군사 행동은 어느 쪽이 당했든 부정**입니다.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 기준: 러시아 입장이 아니라 '한국 언론이 이 사안을 어떻게 평가하는가'
#   긍정 = 한국 언론이 좋게 평가 (협력·평화·관계 개선·경제 기회)
#   부정 = 한국 언론이 나쁘게 평가 (위협·도발·우려·리스크)
#          ※ 북한과 러시아의 협력은 한국 입장에서 위협이므로 부정
#   중립 = 평가 없이 사실만 전달 (만났다, 밝혔다, 발표했다)
# 결과를 보면서 단어를 추가/삭제해 다듬으세요
# ══════════════════════════════════════════════════════════════
POS_WORDS = [
    # 협력·관계
    "협력", "관계 개선", "관계 회복", "교류 확대", "우호", "화해", "해빙", "훈풍", "정상화",
    # 평화·안정
    "평화", "휴전 합의", "종전", "평화 협상", "긴장 완화", "안정", "진전", "돌파구",
    # 경제 기회
    "교역 재개", "수출 재개", "직항 재개", "재진출", "복귀", "기회", "호재", "수혜",
    "북극항로", "에너지 협력", "공급 안정", "기대감",
]
NEG_WORDS = [
    # 군사 위협
    "위협", "도발", "침공", "침략", "공습", "폭격", "미사일", "핵위협", "핵 위협", "무력 과시",
    "군사 위협", "점령",
    # 교전·군사 손실 (어느 쪽 피해든 전쟁 지속 → 부정)
    "격추", "드론 공격", "자폭 드론", "요격", "교전", "격전", "전사", "사상자", "사망자",
    "고전", "퇴각", "패퇴", "반격", "포격",
    # 북러 밀착 (한국 안보 위협)
    "파병", "북러 밀착", "북러 협력", "북러 동맹", "군사 기술 이전", "기술 이전", "무기 거래",
    # 우려·비판
    "우려", "경고", "규탄", "비판", "비난", "반발", "강행", "압박", "협박",
    # 리스크·피해
    "리스크", "위기", "불안", "긴장 고조", "피해", "타격", "손실", "억류", "제재", "충격",
    # 협상 실패·갈등
    "결렬", "교착", "무산", "균열", "갈등", "고립",
]
NEUTRAL_WORDS = [  # 점수에 안 들어가고, 사실 전달 기사인지 확인용으로만 표시
    "회담", "만났", "통화", "밝혔", "발표", "방문", "참석", "말했", "전했", "보도", "개최",
]


def classify(title, body):
    body = body[:500]  # 리드 부분만 (뒤쪽은 배경 설명이 많아 노이즈)
    relevant = any(w in title for w in RUSSIA_WORDS) or \
               sum(body.count(w) for w in ["러시아", "푸틴"]) >= 2

    def hits(words):  # 제목 등장은 가중치 2배
        d = {w: title.count(w) * 2 + body.count(w) for w in words}
        return {w: c for w, c in d.items() if c}

    pos, neg_ctx, neu_ctx = pos_hits(title, body, POS_WORDS)   # 긍정어는 뒤 문맥까지 봄
    neg, neu = hits(NEG_WORDS), hits(NEUTRAL_WORDS)
    neg.update(neg_ctx)
    neu.update(neu_ctx)

    # 북한이 기사 중심(제목에 등장하거나 리드에 2번 이상)이면 남은 긍정어도 부정으로 뒤집음
    nk = any(w in title for w in NK_WORDS) or sum(body.count(w) for w in NK_WORDS) >= 2
    if nk and pos:
        for w, c in pos.items():
            neg[f"북한+{w}"] = c
        pos = {}

    score = sum(pos.values()) - sum(neg.values())
    label = "긍정" if score >= 2 else "부정" if score <= -2 else "중립"
    return pd.Series({"relevant": relevant, "nk": nk, "score": score, "label": label,
                      "pos_words": ", ".join(pos), "neg_words": ", ".join(neg),
                      "neu_words": ", ".join(neu)})


## 2-2. 실행
대상: `NEWS_SECTIONS` 섹션 × `NEWS_PRESS` 언론사.
시사·문화 어느 쪽에도 없는 섹션(IT/과학, 미분류 등)은 **기타**로 같은 기준을 적용합니다. 기타에 러시아 관련 기사가 없으면 표시하지 않습니다.


In [ ]:
news = run_part("시사", classify, NEWS_SECTIONS, NEWS_PRESS)

# 기타: 시사·문화 어느 쪽 섹션에도 없는 것(IT/과학, 미분류 등). 러시아 관련 기사가 없으면 아무것도 안 나옴
etc_sections = sorted(set(load_articles()["section"]) - set(NEWS_SECTIONS) - set(CULTURE_SECTIONS))
etc = run_part("기타", classify, etc_sections, NEWS_PRESS, hide_empty=True)


---
# Part 3. 연예·스포츠·생활·문화 분석
## 3-1. 문화 단어 사전
시사 사전은 정치·군사 단어 위주라 문화 기사는 대부분 "러시아 관련 아님"으로 빠지거나 중립이 됩니다. 그래서 문화 전용 단어로 따로 봅니다.
- **관련 판단**: 제목에 러시아 관련어가 있거나, 본문 **전체**에서 러시아·러시아 문화 관련어(차이콥스키, 볼쇼이, 상트페테르부르크 등)가 2번 이상 나오면 관련 기사로 봅니다. "러시아 출신 인플루언서"처럼 한 번 스치는 기사는 제외합니다.
- **긍정**: 내한·협연·매진·호평·수상·우승·문화 교류처럼 좋게 평가
- **부정**: 공연 취소·보이콧·퇴출·출전 금지·도핑·친푸틴 논란·검열처럼 나쁘게 평가 (전쟁 때문에 생긴 문화·스포츠계 갈등 포함)


In [ ]:
CULTURE_RUSSIA_WORDS = RUSSIA_WORDS + [
    "상트페테르부르크", "볼쇼이", "마린스키", "차이콥스키", "라흐마니노프",  # '러시아어'는 '러시아'와 이중으로 세져서 제외
    "쇼스타코비치", "프로코피예프", "스트라빈스키", "무소륵스키", "톨스토이", "도스토옙스키",
    "체호프", "푸시킨", "고골", "나보코프", "칸딘스키", "말레비치", "타르콥스키",
]
CULTURE_POS = [
    # 공연·전시
    "내한", "협연", "초연", "매진", "흥행", "열풍", "화제", "인기",
    # 평가
    "호평", "극찬", "찬사", "감동", "거장", "걸작", "명작", "전설적", "수상",
    # 스포츠
    "우승", "금메달", "메달", "신기록", "출전 허용", "징계 해제",
    # 교류
    "문화 교류", "교류", "공동 제작", "합작", "재개", "복귀",
]
CULTURE_NEG = [
    # 전쟁으로 인한 문화계·스포츠계 갈등
    "보이콧", "퇴출", "배제", "출전 금지", "참가 금지", "자격 정지", "공연 취소", "취소", "불참", "하차",
    "친푸틴", "친러", "전쟁", "침공", "제재",
    # 억압·논란
    "검열", "탄압", "체포", "망명", "논란", "비판", "반발", "도핑", "약물", "학대", "징계",
]


def classify_culture(title, body):
    relevant = any(w in title for w in CULTURE_RUSSIA_WORDS) or \
               sum(body.count(w) for w in CULTURE_RUSSIA_WORDS) >= 2
    lead = body[:500]

    def hits(words):  # 제목 등장은 가중치 2배
        d = {w: title.count(w) * 2 + lead.count(w) for w in words}
        return {w: c for w, c in d.items() if c}

    pos, neg_ctx, neu = pos_hits(title, lead, CULTURE_POS)
    neg = hits(CULTURE_NEG)
    neg.update(neg_ctx)

    nk = any(w in title for w in NK_WORDS) or sum(lead.count(w) for w in NK_WORDS) >= 2
    if nk and pos:
        for w, c in pos.items():
            neg[f"북한+{w}"] = c
        pos = {}

    score = sum(pos.values()) - sum(neg.values())
    label = "긍정" if score >= 2 else "부정" if score <= -2 else "중립"
    return pd.Series({"relevant": relevant, "nk": nk, "score": score, "label": label,
                      "pos_words": ", ".join(pos), "neg_words": ", ".join(neg),
                      "neu_words": ", ".join(neu)})


## 3-2. 실행
대상: `CULTURE_SECTIONS` 섹션 × `CULTURE_PRESS` 언론사.


In [ ]:
culture = run_part("문화·생활", classify_culture, CULTURE_SECTIONS, CULTURE_PRESS)
